## Pytorch minimal workshop

### How to load data
First, import crucial dependencies

In [1]:

import numpy as np
import pandas as pd

import torch

# If you want to run PyTorch on GPU, you need to run a different torch setup 
# (now defined as additional dependency, to be activated by uv sync --extra cu126 
# This is highly dependent on your architecture!! Let me know if that is something of interest!

# print(torch.cuda.is_available())
# print(torch.cuda.get_device_name(0))





A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/hassanhaydar/DSA104/.venv/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/hassanhaydar/DSA104/.venv/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/Users/hassanhaydar/DSA104/.venv/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 758, in start
    self.io_lo

Load a dataset using pandas. Pandas is still quite convenient for working with data, but ultimately, pytorch needs tensors (which are, just like DataFrames, built on top of numpy arrays). Any datasplits should also be done before transforming the dataframes into tensors.

In [2]:
from sklearn.datasets import load_diabetes
import pandas as pd

data = load_diabetes()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.DataFrame(data.target)

print(y.shape)
print(X.head())

(442, 1)
        age       sex       bmi        bp        s1        s2        s3  \
0  0.038076  0.050680  0.061696  0.021872 -0.044223 -0.034821 -0.043401   
1 -0.001882 -0.044642 -0.051474 -0.026328 -0.008449 -0.019163  0.074412   
2  0.085299  0.050680  0.044451 -0.005670 -0.045599 -0.034194 -0.032356   
3 -0.089063 -0.044642 -0.011595 -0.036656  0.012191  0.024991 -0.036038   
4  0.005383 -0.044642 -0.036385  0.021872  0.003935  0.015596  0.008142   

         s4        s5        s6  
0 -0.002592  0.019907 -0.017646  
1 -0.039493 -0.068332 -0.092204  
2 -0.002592  0.002861 -0.025930  
3  0.034309  0.022688 -0.009362  
4 -0.002592 -0.031988 -0.046641  


There are different ways to convert a dataframe into a tensor:

1) Directly from dataframe, via the `values` property.

In [4]:
X_tensor = torch.tensor(X.values)
X_tensor #is just a version of numpy array, but in torch format, which can be used for deep learning models.

tensor([[ 0.0381,  0.0507,  0.0617,  ..., -0.0026,  0.0199, -0.0176],
        [-0.0019, -0.0446, -0.0515,  ..., -0.0395, -0.0683, -0.0922],
        [ 0.0853,  0.0507,  0.0445,  ..., -0.0026,  0.0029, -0.0259],
        ...,
        [ 0.0417,  0.0507, -0.0159,  ..., -0.0111, -0.0469,  0.0155],
        [-0.0455, -0.0446,  0.0391,  ...,  0.0266,  0.0445, -0.0259],
        [-0.0455, -0.0446, -0.0730,  ..., -0.0395, -0.0042,  0.0031]])

2. Via `torch.tensor` but via numpy array. Likewise, numpy arrays can be directly converted into tensors.

In [5]:
X_tensor = torch.tensor(X.to_numpy(), dtype=torch.float32) # datatype conversion optional
X_tensor

tensor([[ 0.0381,  0.0507,  0.0617,  ..., -0.0026,  0.0199, -0.0176],
        [-0.0019, -0.0446, -0.0515,  ..., -0.0395, -0.0683, -0.0922],
        [ 0.0853,  0.0507,  0.0445,  ..., -0.0026,  0.0029, -0.0259],
        ...,
        [ 0.0417,  0.0507, -0.0159,  ..., -0.0111, -0.0469,  0.0155],
        [-0.0455, -0.0446,  0.0391,  ...,  0.0266,  0.0445, -0.0259],
        [-0.0455, -0.0446, -0.0730,  ..., -0.0395, -0.0042,  0.0031]])

3. Using `DataLoader` for large datasets (data larger than allocated memory)

In [6]:
from torch.utils.data import DataLoader, TensorDataset

X_tensor = torch.tensor(X.to_numpy())
y_tensor = torch.tensor(y.to_numpy())

dataset = TensorDataset(X_tensor, y_tensor)

dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

for batch in dataloader:
    print(batch)

[tensor([[-0.0055, -0.0446,  0.0240,  0.0081, -0.0346, -0.0389,  0.0229, -0.0395,
         -0.0160, -0.0135],
        [ 0.0962, -0.0446,  0.0401, -0.0573,  0.0452,  0.0607, -0.0213,  0.0362,
          0.0126,  0.0238]]), tensor([[121.],
        [180.]])]
[tensor([[ 0.0018,  0.0507,  0.0110, -0.0194, -0.0167, -0.0038, -0.0471,  0.0343,
          0.0241,  0.0238],
        [-0.0382, -0.0446,  0.0671, -0.0608, -0.0291, -0.0232, -0.0103, -0.0026,
         -0.0015,  0.0196]]), tensor([[111.],
        [ 78.]])]
[tensor([[ 0.0453,  0.0507, -0.0030,  0.1079,  0.0356,  0.0225,  0.0266, -0.0026,
          0.0280,  0.0196],
        [-0.0782, -0.0446, -0.0170, -0.0126, -0.0002, -0.0135,  0.0707, -0.0395,
         -0.0412, -0.0922]]), tensor([[217.],
        [ 90.]])]
[tensor([[ 0.0308,  0.0507,  0.0326,  0.0494, -0.0401, -0.0436, -0.0692,  0.0343,
          0.0630,  0.0031],
        [-0.0491, -0.0446,  0.1609, -0.0470, -0.0291, -0.0198, -0.0471,  0.0343,
          0.0280,  0.0113]]), tensor([[208.]

4. Use a Custom Dataset Class for more flexibility and cleaner code:

In [7]:

from torch.utils.data import Dataset, DataLoader

class MyDataFromDF(Dataset):
    def __init__(self, X, y):
        # features provided as Dataframes converted
        self.X = torch.tensor(    
            X.to_numpy(), 
            # dtype=torch.float32() # optionally include a datatype
            ) 
        
        # targets provided as Dataframes
        self.y = torch.tensor( 
            y.to_numpy(),
            # dtype=torch.float32() # optionally include a datatype
            )            

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]   # always return (features, target)


In [8]:
# Wrap custom dataset in Dataloader

dataset = MyDataFromDF(X, y)
loader = DataLoader(dataset, batch_size=2, shuffle=True)

# for batch in loader:
#     print(batch)

In [9]:
# How to use the loader in training:
for batch_X, batch_y in loader:
    pred = model(batch_X)
    loss = criterion(pred, batch_y)
    ...


NameError: name 'model' is not defined

Datatypes need to be handled by torch (integer, float, Boolean) - compatible dtypes will be converted into a common format (check with `print(tensor.dtype)`). A datatype can be specified in (`dtype=torch.float32`) - see some examples above.

Strings, categorical values are incompatible and need to be encoded into numerical values.

## Build a NN for Regression
For convenience and better comparison, the Diabetes dataset as available in scikit-learn is used here again.

Train-test split on original data:

In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

Convert to tensors - you could also use any other method from above! Note that here there was no DataLoader used as wrapper, as the dataset is small.

In [11]:
X_train = torch.tensor(X_train.to_numpy(), dtype=torch.float32)
y_train = torch.tensor(y_train.to_numpy(), dtype=torch.float32)
X_test = torch.tensor(X_test.to_numpy(), dtype=torch.float32)
y_test = torch.tensor(y_test.to_numpy(), dtype=torch.float32)

A fully connected NN is defined as model class. Note the separation of the initialisation and the forward pass class functions (essentially separating the learning part from the activation functions).

In [12]:
import torch.nn as nn

class DiabetesNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(DiabetesNN, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)  # input layer with width = length of the feature set
        self.fc2 = nn.Linear(hidden_size, hidden_size)  # one hidden layer, try to add another one
        self.fc3 = nn.Linear(hidden_size, output_size)  # output layer
    
    # Good practice: activation functions specified in forward pass - separated from the learning parts
    def forward(self, x):            # forward pass
        x = torch.relu(self.fc1(x))  # ReLU activation function for input layer
        x = torch.relu(self.fc2(x))  # ReLU activation function for hidden layer
        # no activation function for the output layer (because it is a regresion model!)
        return x

Define hyperparameters:

In [13]:
# Hyperparameters
input_size = X_train.shape[1]
hidden_size = 20
output_size = 1
learning_rate = 0.05

# Number of training iterations
num_epochs = 100

Introduce loss function (``criterion``) and gradient optimiser (``optimizer``):

In [14]:
import torch.optim as optim
model = DiabetesNN(input_size, hidden_size, output_size)

criterion = nn.MSELoss()  # loss function defined;

optimizer = optim.Adam(model.parameters(), learning_rate) # gradient descent method based on average and squares of gradient

In [15]:
for epoch in range(num_epochs):
    
    model.train()
    outputs = model(X_train)
    loss = criterion(outputs, y_train)

    optimizer.zero_grad() # clear existing gradients
    loss.backward() # backpropagation of loss function to optimise gradient
    optimizer.step() # update parameters using SDG

    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch + 1}/100], Loss: {loss.item():.4f}')

Epoch [10/100], Loss: 28865.4863
Epoch [20/100], Loss: 25900.7461
Epoch [30/100], Loss: 20361.2188
Epoch [40/100], Loss: 13505.4678
Epoch [50/100], Loss: 8180.5220
Epoch [60/100], Loss: 6515.0889
Epoch [70/100], Loss: 6466.5645
Epoch [80/100], Loss: 6118.0464
Epoch [90/100], Loss: 5952.4111
Epoch [100/100], Loss: 5879.0493


/Users/hassanhaydar/DSA104/.venv/lib/python3.12/site-packages/torch/nn/modules/loss.py:535: UserWarning: Using a target size (torch.Size([353, 1])) that is different to the input size (torch.Size([353, 20])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Evaluate the model on the test set to termine the generalisation for the regression model.

In [16]:
# Evaluate the model
mse_loss = nn.MSELoss()

model.eval()

with torch.no_grad():
    y_pred = model(X_test)
    mse = mse_loss(y_pred, y_test).item()
    print("MSE:", mse)


MSE: 5415.83349609375


/Users/hassanhaydar/DSA104/.venv/lib/python3.12/site-packages/torch/nn/modules/loss.py:535: UserWarning: Using a target size (torch.Size([89, 1])) that is different to the input size (torch.Size([89, 20])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
